In [ ]:
import numpy as np
import shtns
import matplotlib.pyplot as plt
from tqdm import tqdm

# 初始化 shtns 对象
lmax = 1020
sht = shtns.sht(lmax)
nlats, nlons = sht.set_grid()  # 自动选择合适网格
degree = sht.l
print("网格大小:", nlats, nlons, nlats * nlons)
lats = np.arcsin(sht.cos_theta)
lons = (2.*np.pi/nlons)*np.arange(nlons)
theta_m, phi_m = np.meshgrid(lats, lons, indexing='ij')

# 随机初始化网格数据
grid = np.random.randn(nlats, nlons)

lap = -degree*(degree+1.0).astype(complex)
invlap = np.zeros(lap.shape, lap.dtype)
invlap[1:] = 1./lap[1:]
spec_filter = np.exp(-36.0 * (degree / lmax)**8.0)  # 高斯滤波器

# 计算能量谱斜率
slope = np.ones_like(degree, dtype=complex)
slope[degree > 0] = degree[degree > 0]**(-1/3)

# 初始化球谐网格（高斯-勒让德）和初始ψ、ζ
zeta_coeffs = sht.analys(grid) * slope  # SHCoeffs, 初始 streamfunction
zeta_coeffs *= spec_filter  # 应用高斯滤波器
psi_coeffs = zeta_coeffs.copy()  # ζ = ∇²ψ
psi_coeffs *= invlap

dt = 0.5  # 时间步长
nu = 0.000000001  # 粘性系数

In [ ]:
def streamfunction_from_zeta(zeta_coeffs):
    """从 ζ 系数计算 ψ 系数"""
    psi_coeffs = zeta_coeffs.copy()
    psi_coeffs *= invlap  # ψ = ζ / -l(l+1)
    return psi_coeffs

def velocity_from_streamfunction(psi_coeffs):
    """从 ψ 系数计算速度场 u"""
    # 速度场 u = (u_theta, u_phi)
    # 注意球面旋度流的方向, u_θ = 1/sinθ*∂ψ/∂φ
    # u_φ = -∂ψ/∂θ
    u_phi, u_theta = sht.synth_grad(psi_coeffs) 
    return u_phi, u_theta

def velocity_from_zeta(zeta_coeffs):
    """从 ζ 系数计算速度场 u"""
    # 先计算 ψ
    psi_coeffs = streamfunction_from_zeta(zeta_coeffs)
    return velocity_from_streamfunction(psi_coeffs)

def nonl(zeta_coeffs):
    # 更新 ψ：解 ∇²ψ = ζ，即 ψ = ζ / -l(l+1)
    psi_coeffs = streamfunction_from_zeta(zeta_coeffs)

    # 速度场 u = (u_theta, u_phi)
    # 注意球面旋度流的方向, u_θ = 1/sinθ*∂ψ/∂φ
    # u_φ = -∂ψ/∂θ
    u_phi, u_theta = velocity_from_streamfunction(psi_coeffs)
    
    # 计算非线性项 N = u ⋅ ∇ζ
    #  -∂/∂θ 和 1/sinθ*∂/∂φ
    dtheta, dphi = sht.synth_grad(zeta_coeffs) 
    dtheta *= -1
    adv = u_theta * dtheta + u_phi * dphi

    # 回到频域（快速球面变换）
    adv_coeffs = sht.analys(adv)  # 转换为球谐系数
    adv_coeffs *= spec_filter

    return - adv_coeffs + nu * (zeta_coeffs * lap)

In [ ]:
sol = [sht.synth(zeta_coeffs)]  # 存储每个时间步的ζ系数
energy = [np.sum(np.abs(sht.synth(zeta_coeffs)*sht.synth(psi_coeffs)))]
e0 = energy[0]  # 初始能量
num_steps = 2000  # 时间步数
with tqdm(total=num_steps, desc="时间推进") as pbar:
    for t in range(num_steps):
        # 时间推进：Euler 或 Runge-Kutta
        k1 = dt * nonl(zeta_coeffs)
        k2 = dt * nonl(zeta_coeffs + 0.5 * k1)
        k3 = dt * nonl(zeta_coeffs + 0.5 * k2)
        k4 = dt * nonl(zeta_coeffs + k3)
        zeta_coeffs += (k1 + 2*k2 + 2*k3 + k4) / 6.0

        # 存储当前时间步的 ζ 系数
        if (t+1) % 10 == 0:
            # 存储当前 ζ 系数
            sol.append(sht.synth(zeta_coeffs))
            energy.append(np.sum(np.abs(sht.synth(zeta_coeffs)*sht.synth(streamfunction_from_zeta(zeta_coeffs)))))
            pbar.set_postfix({"能量": energy[-1] / e0})
            pbar.update(10)

In [ ]:
plt.plot(energy, label='能量')

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 100  # Allow up to 50 MB

fig, ax = plt.subplots(figsize=(8, 4))
u, v = velocity_from_zeta(sht.analys(sol[0]))
umag = np.sqrt(u**2 + v**2)
im = ax.imshow(umag, origin='lower', cmap='RdBu_r', extent=[0, 360, -90, 90], aspect='auto')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Velocity Evolution (step=0)')
plt.colorbar(im, ax=ax, label='U Magnitude')


def update(frame):
    u, v = velocity_from_zeta(sht.analys(sol[frame]))
    umag = np.sqrt(u**2 + v**2)
    im.set_data(umag)
    ax.set_title(f'Velocity Evolution (step={frame})')
    return [im]

ani = FuncAnimation(fig, update, frames=len(sol), interval=50, blit=False, repeat=False)

HTML(ani.to_jshtml())


In [ ]:
import cartopy.crs as ccrs


fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.Orthographic(0, 0))
ax.set_global()
#ax.coastlines()
ax.set_facecolor('black')

xs, ys = np.meshgrid(np.rad2deg(lons), np.flip(np.rad2deg(lats)), indexing='ij')

u, v = velocity_from_zeta(sht.analys(sol[0]))
umag = np.sqrt(u**2 + v**2)
zs = umag.T  # 使用速度场的大小作为颜色映射
mesh = ax.pcolormesh(xs, ys, zs, transform=ccrs.PlateCarree(),
                     shading='auto')


def update_mesh(t):
    u, v = velocity_from_zeta(sht.analys(sol[t]))
    umag = np.sqrt(u**2 + v**2)
    #umag = np.roll(umag, t % 360, axis=1)
    mesh.set_array(umag.T.flatten())


ani = FuncAnimation(fig, update_mesh, frames=len(sol),
                    interval=100, blit=False)

HTML(ani.to_jshtml())